In [ ]:
!pip install datasets tqdm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

## setup

In [ ]:
ds = load_dataset("roneneldan/TinyStories")
ds

In [ ]:
train_text = "\n\n".join(ds["train"]["text"])
val_text = "\n\n".join(ds["validation"]["text"])

print("Train chars:", len(train_text))
print("Val chars:", len(val_text))

In [ ]:
chars = sorted(list(set(train_text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

print("Vocab size:", vocab_size)
print(chars[:100])

In [ ]:
train_ids = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_ids = torch.tensor([stoi.get(c, 0) for c in val_text], dtype=torch.long)

print(train_ids.shape, val_ids.shape)

## data loader

In [ ]:
class CharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


block_size = 128

train_dataset = CharDataset(train_ids, block_size)
val_dataset = CharDataset(val_ids, block_size)

len(train_dataset), len(val_dataset)

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

In [ ]:
## model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.block_size = block_size

        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)  # (B, T, 3C)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, H, T, T)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v  # (B, H, T, D)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)

        y = self.out_proj(y)
        y = self.resid_dropout(y)
        return y


class MLP(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            block_size=block_size,
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model=d_model, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class BaselineModel(nn.Module):
    """
    Simple character-level causal transformer for next-token prediction.
    Input:  x of shape (B, T)
    Output: logits of shape (B, T, vocab_size)
    """

    def __init__(
        self,
        vocab_size,
        block_size=128,
        d_model=256,
        n_heads=4,
        n_layers=4,
        dropout=0.1,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                block_size=block_size,
            )
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x, targets=None):
        B, T = x.shape
        assert T <= self.block_size, f"Sequence length {T} exceeds block_size {self.block_size}"

        pos = torch.arange(0, T, device=x.device)

        tok_emb = self.token_embedding(x)                 # (B, T, d_model)
        pos_emb = self.position_embedding(pos)[None, :, :]  # (1, T, d_model)

        h = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            h = block(h)

        h = self.ln_f(h)
        logits = self.lm_head(h)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, self.vocab_size),
                targets.view(B * T),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """
        idx: (B, T)
        returns: (B, T + max_new_tokens)
        """
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)

        return idx

### train

In [ ]:
import math
import time
import torch


def evaluate(model, loader, device, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch_idx, (xb, yb) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            xb = xb.to(device)
            yb = yb.to(device)

            _, loss = model(xb, yb)

            batch_tokens = xb.numel()
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    return avg_loss, ppl


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=5,
    lr=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    eval_every=500,
    max_train_steps=None,
    val_max_batches=50,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    train_losses = []
    val_losses = []
    val_ppls = []

    step = 0
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()

        running_loss = 0.0
        running_tokens = 0

        for xb, yb in train_loader:
            if max_train_steps is not None and step >= max_train_steps:
                break

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            _, loss = model(xb, yb)
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            batch_tokens = xb.numel()
            running_loss += loss.item() * batch_tokens
            running_tokens += batch_tokens
            step += 1

            if step % eval_every == 0:
                train_loss = running_loss / running_tokens
                train_ppl = math.exp(train_loss)

                val_loss, val_ppl = evaluate(
                    model,
                    val_loader,
                    device=device,
                    max_batches=val_max_batches,
                )

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                val_ppls.append(val_ppl)

                print(
                    f"epoch={epoch+1} step={step} "
                    f"train_loss={train_loss:.4f} train_ppl={train_ppl:.2f} "
                    f"val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}"
                )

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), "baseline_best.pt")
                    print("saved best model to baseline_best.pt")

        epoch_time = time.time() - epoch_start
        epoch_train_loss = running_loss / running_tokens
        epoch_train_ppl = math.exp(epoch_train_loss)

        val_loss, val_ppl = evaluate(
            model,
            val_loader,
            device=device,
            max_batches=val_max_batches,
        )

        print(
            f"[epoch {epoch+1} done] "
            f"time={epoch_time:.1f}s "
            f"train_loss={epoch_train_loss:.4f} train_ppl={epoch_train_ppl:.2f} "
            f"val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "baseline_best.pt")
            print("saved best model to baseline_best.pt")

        if max_train_steps is not None and step >= max_train_steps:
            break

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_ppls": val_ppls,
        "best_val_loss": best_val_loss,
    }

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = BaselineModel(
    vocab_size=vocab_size,
    block_size=128,
    d_model=256,
    n_heads=4,
    n_layers=4,
    dropout=0.1,
).to(device)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=5,
    lr=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    eval_every=300,
    val_max_batches=20,
)